# Lesson 07 — Scale & Cost

You've built the pipeline. Now let's answer the real questions:

- **How do I process 10 videos without submitting 10 jobs manually?** → Array jobs
- **How much does this actually cost?** → Spot pricing math
- **When is GPU worth it vs just using CPU?** → Rule of thumb

No new ML concepts — this lesson is about production thinking.

## Concept: Batch Array Jobs

An **array job** is one submission that fans out into N parallel copies:

```
submit_job(arrayProperties={"size": 5})
  → 5 containers run simultaneously
  → each gets AWS_BATCH_JOB_ARRAY_INDEX = 0, 1, 2, 3, 4
  → each reads its index and picks which video to process
```

Wall-clock time for 5 videos = wall-clock time for 1 video (they run in parallel!).
Cost = 5× a single run.

## Step 1 — Load environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../../.env")
S3_BUCKET = os.environ["S3_BUCKET"]
print(f"Bucket: {S3_BUCKET}")

## Step 2 — Upload sample videos

For this demo we'll use 3 copies of `assets/sample.mp4` to simulate 3 different videos.
In a real scenario you'd have different video files.

In [ ]:
import boto3

s3 = boto3.client("s3")

# Upload the same sample video under 3 different keys to simulate 3 videos
video_keys = []
for i in range(1, 4):
    key = f"videos/clip{i:02d}.mp4"
    s3.upload_file("assets/sample.mp4", S3_BUCKET, key)
    video_keys.append(key)
    print(f"Uploaded → s3://{S3_BUCKET}/{key}")

print(f"\n{len(video_keys)} videos ready.")

## Step 3 — Submit the array job

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "submit_array_job.py", "--video-keys"] + video_keys,
)
print("Exit code:", result.returncode)

## Step 4 — Cost breakdown

In [ ]:
# Approximate numbers for g4dn.xlarge Spot in ap-northeast-1
SPOT_PRICE_PER_HOUR  = 0.20    # USD, typical Spot price (range $0.16–0.24)
EXTRACT_MINUTES      = 1.5     # per video
EMBED_MINUTES        = 2.5     # per video
PIPELINE_MINUTES     = EXTRACT_MINUTES + EMBED_MINUTES
N_VIDEOS             = len(video_keys)

# With an array job, all videos run in parallel → wall-clock = 1 pipeline run
# But we pay for N instances
cost_per_instance = (PIPELINE_MINUTES / 60) * SPOT_PRICE_PER_HOUR
total_cost        = cost_per_instance * N_VIDEOS
wall_clock_min    = PIPELINE_MINUTES   # parallel, so same as 1

print(f"Videos processed        : {N_VIDEOS}")
print(f"Wall-clock time         : ~{wall_clock_min:.0f} min  (parallel — same as 1 video!)")
print(f"Cost per instance       : ${cost_per_instance:.4f}")
print(f"Total cost ({N_VIDEOS} videos)  : ${total_cost:.4f}")
print()
print(f"At this rate, 100 videos would cost: ${cost_per_instance * 100:.2f}")

## Step 5 — GPU vs CPU: when does it pay off?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Hypothetical: embed N frames using CPU (300 ms/frame) vs GPU (5 ms/frame)
n_frames    = np.array([10, 50, 100, 500, 1000, 5000, 10000])
cpu_seconds = n_frames * 0.30   # 300 ms/frame on CPU
gpu_seconds = n_frames * 0.005  # 5 ms/frame on GPU (batch of 16)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(n_frames, cpu_seconds / 60, label="CPU",  color="steelblue", marker="o")
ax.plot(n_frames, gpu_seconds / 60, label="GPU",  color="coral",     marker="o")
ax.set_xscale("log")
ax.set_xlabel("Number of frames")
ax.set_ylabel("Time (minutes)")
ax.set_title("GPU vs CPU time for CLIP embedding")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Speedup at different scales:")
for n, c, g in zip(n_frames, cpu_seconds, gpu_seconds):
    print(f"  {n:6d} frames — CPU: {c:6.1f}s  GPU: {g:4.1f}s  Speedup: {c/g:.0f}×")

## 🎉 Course Complete!

Here's what you built — a production-grade GPU pipeline:

```
Video in S3
    ↓  [GPU: extract_frames.py]     → frames in S3
    ↓  [GPU: embed_frames.py]       → CLIP embeddings in S3
    ↓  [local: search notebook]     → "find me the frame with a dog" ✅
       [scale: array job]           → process 100 videos in parallel ✅
```

### What to explore next

| Idea | What to look up |
|------|-----------------|
| Replace numpy search with a real vector DB | FAISS, pgvector, Pinecone |
| Transcribe audio from video | OpenAI Whisper (runs on GPU too!) |
| Caption frames automatically | BLIP-2, LLaVA |
| Orchestrate bigger pipelines | Apache Airflow, AWS Step Functions |
| Train your own embedding model | PyTorch fine-tuning, PEFT/LoRA |